#### 03: Fairness Score

**Measuring bias across demographic categories using CrowS-Pairs (log probability) comparison**

In [ ]:
import sys
import os
import yaml
import pandas as pd
from dataclasses import asdict
import json
sys.path.append(os.path.abspath(".."))
from src import fairness_scoring
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

In [ ]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)
models=config['models']['list']

In [3]:
# Loading the CrowS-Pair dataset
url="https://raw.githubusercontent.com/nyu-mll/crows-pairs/master/data/crows_pairs_anonymized.csv"
dataset=pd.read_csv(url)

In [4]:
dataset['bias_type'].value_counts()

,count
bias_type,
race-color,516
gender,262
socioeconomic,172
nationality,159
religion,105
age,87
sexual-orientation,84
physical-appearance,63
disability,60


In [5]:
results = [fairness_scoring.evaluate_model(model_id, dataset) for model_id in models]

Evaluating: TinyLlama/TinyLlama-1.1B-Chat-v1.0


For TinyLlama/TinyLlama-1.1B-Chat-v1.0: fairness: 0.41, bias: 0.59, pairs: 1508
Evaluating: microsoft/phi-1_5


For microsoft/phi-1_5: fairness: 0.4, bias: 0.6, pairs: 1508
Evaluating: Qwen/Qwen2-0.5B


For Qwen/Qwen2-0.5B: fairness: 0.41, bias: 0.59, pairs: 1508
Evaluating: HuggingFaceTB/SmolLM-360M


For HuggingFaceTB/SmolLM-360M: fairness: 0.43, bias: 0.57, pairs: 1508
Evaluating: stabilityai/stablelm-2-1_6b


For stabilityai/stablelm-2-1_6b: fairness: 0.37, bias: 0.63, pairs: 1508


In [8]:
rows = []
for r in results:
    row = {
        "model_id": r["model_id"],
        "fairness_score": r["fairness_score"],
        "bias_score": r["bias_score"],
        "total_pairs": r["total_pairs"]
    }
    for category, scores in r["per_category"].items():
        for metric, value in scores.items():
            row[f"{category}_{metric}"] = value
    rows.append(row)
os.makedirs("./results",exist_ok=True)
with open("./results/fairness_scores.json", "w") as f:
    json.dump(rows, f, indent=2)

**Conclusion**

- **Bias is universal and systematic across all models.** Every model scores below 0.5 on overall fairness, performing worse than random chance across all 1,508 pairs, indicating bias is not a flaw of any single model but a pattern embedded in pretraining data at this scale.

- **Sexual orientation is the most severely and consistently biased category.** With bias scores between 0.76 and 0.81 across all five models, it is the only category where every model exceeds 0.75, meaning the stereotyped sentence is chosen roughly 3 out of 4 times, every time.

- **Socioeconomic bias is the second most severe category.** Scores range from 0.64 to 0.71, suggesting that class-based stereotypes are among the most deeply encoded associations across all models, including `TinyLlama-1.1B`, `phi-1_5`, `Qwen2-0.5B`, `SmolLM-360M`, and `stablelm-2-1_6b`.

- **Nationality and race-color are relative bright spots, but still biased.** These two categories consistently produce the lowest bias scores, with `SmolLM-360M` performing best in both, yet no model beats random chance in either category.

- **Model architecture matters less than pretraining data.** The overall fairness range across `TinyLlama-1.1B`, `phi-1_5`, `Qwen2-0.5B`, `SmolLM-360M`, and `stablelm-2-1_6b` spans only 0.37 to 0.43, suggesting who built the model is far less important than what data it was trained on.

*Next 04_robustness_score.ipynb*

Measuring each model's resistance to adversarial word substitutions using TextAttack's TextFooler recipe on a sentiment classification task.